# Congested Traffic Policy

Train and evaluate seven DQN policy variants on the same congested traffic distribution. Each experiment trains for 20k steps, then evaluates the saved model for 1,000 episodes with the standard speed, TTC, collision, overtake, reward, and adaptive/safety metrics.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

from dqn_notebook_utils import (
    adaptive_metric_columns,
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    train_and_display,
)

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "congested_traffic_policy"
RESULTS_SUBDIR = "congested_traffic_policy"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "dqn" / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook dir:", NOTEBOOK_DIR)
print("Results dir:", RESULTS_DIR)

## Shared Congested Traffic Configuration

In [ ]:
# Shared congested traffic setup for all seven experiments.
# Congestion is represented by high vehicle density plus a continuous spectrum of IDM/MOBIL driver behavior.
environment_profile = "structured_baseline"
eval_environment_profile = environment_profile

congested_environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 45,
    "duration": 40,
    "ego_spacing": 1.0,
    "vehicles_density": 2.0,
    "simulation_frequency": 15,
    "policy_frequency": 1,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}
congested_eval_environment_overrides = dict(congested_environment_overrides)

congested_observation_config = {
    "vehicles_count": 12,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}
action_config = {
    "type": "DiscreteMetaAction",
}
base_reward_config = {
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "normalize_reward": True,
}
speed_config = {
    "reward_speed_range": [20.0, 30.0],
}

# Continuous surrounding-driver behavior: 0 blue/conservative, 100 red/aggressive.
congested_driver_aggressiveness_config = {
    "enabled": True,
    "distribution": "uniform",
    "min_score": 0.0,
    "max_score": 100.0,
    "fixed_score": None,
    "normal_mean": 50.0,
    "normal_std": 20.0,
    "conservative": {
        "target_speed": 18.0,
        "acc_max": 4.0,
        "comfort_acc_max": 2.0,
        "comfort_acc_min": -4.0,
        "delta": 4.5,
        "time_wanted": 2.4,
        "distance_wanted": 14.0,
        "politeness": 0.8,
        "lane_change_min_acc_gain": 0.8,
        "lane_change_max_braking_imposed": 1.0,
        "lane_change_delay": 1.5,
    },
    "aggressive": {
        "target_speed": 30.0,
        "acc_max": 7.0,
        "comfort_acc_max": 5.5,
        "comfort_acc_min": -6.5,
        "delta": 3.5,
        "time_wanted": 0.6,
        "distance_wanted": 6.0,
        "politeness": 0.0,
        "lane_change_min_acc_gain": 0.05,
        "lane_change_max_braking_imposed": 3.5,
        "lane_change_delay": 0.5,
    },
}

safety_ttc_flow_reward_config = {
    "enabled": True,
    "ttc_safe_threshold": 4.0,
    "ttc_target": 6.0,
    "ttc_cap": 10.0,
    "low_ttc_penalty_weight": 0.7,
    "max_low_ttc_penalty": 0.9,
    "safe_ttc_bonus_weight": 0.08,
    "max_safe_ttc_bonus": 0.12,
    "lag_penalty_weight": 0.16,
    "speed_tolerance": 2.0,
    "max_lag_penalty": 0.9,
    "rear_ttc_pressure": 5.0,
    "rear_pressure_floor": 0.25,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}

adaptive_wide_band_config = {
    "enabled": True,
    "mode": "delta",
    "ttc_midpoint": 4.0,
    "ttc_temperature": 0.75,
    "ttc_cap": 10.0,
    "min_target_speed": 20.0,
    "max_target_speed": 30.0,
    "faster_max_delta": 10.0,
    "slower_min_delta": 0.0,
    "slower_max_delta": 10.0,
}

attention_config = {
    "features_dim": 64,
    "attention_heads": 2,
    "attention_dropout": 0.0,
    "presence_feature_idx": 0,
    "embedding_arch": "64,64",
    "net_arch": "64,64",
}

timesteps = 20_000
saved_model_eval_episodes = 1000
eval_episodes_during_training = 5
n_envs = min(4, os.cpu_count() or 1)
seed = 42
train_freq = 4
gradient_steps = train_freq * n_envs
saved_model_eval_seed = seed + 10_000
saved_model_eval_name = f"saved_model_eval_{saved_model_eval_episodes}_episodes"

hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.70,
    "exploration_final_eps": 0.10,
    "progress_every": 1_000,
    "verbose": 0,
}

base_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=congested_environment_overrides,
    observation=congested_observation_config,
    action=action_config,
    reward=base_reward_config,
    speed=speed_config,
    adaptive_longitudinal={"enabled": False},
    rear_flow={"enabled": False},
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward={"enabled": False},
    driver_aggressiveness=congested_driver_aggressiveness_config,
)

display(pd.DataFrame({"congested_base_env": pd.Series(base_env_config)}))

## Runner Helpers

In [ ]:
completed_experiments: dict[str, dict] = {}


def make_congested_env_config(
    *,
    safety_reward: bool = False,
    adaptive: bool = False,
) -> dict:
    return build_env_config(
        profile_name=environment_profile,
        profile_overrides=congested_environment_overrides,
        observation=congested_observation_config,
        action=action_config,
        reward=base_reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_wide_band_config if adaptive else {"enabled": False},
        rear_flow={"enabled": False},
        traffic_flow_reward={"enabled": False},
        safety_ttc_flow_reward=safety_ttc_flow_reward_config if safety_reward else {"enabled": False},
        driver_aggressiveness=congested_driver_aggressiveness_config,
    )


def run_congested_experiment(
    *,
    run_name: str,
    label: str,
    backend_module: str,
    safety_reward: bool = False,
    adaptive: bool = False,
    attention: bool = False,
    seed_offset: int = 0,
) -> dict:
    trainer, _, _, results_dir, default_device = load_dqn_backend(
        backend_module=backend_module,
        notebook_subdir="congested_traffic_policy",
        results_subdir=RESULTS_SUBDIR,
    )
    env_config = make_congested_env_config(
        safety_reward=safety_reward,
        adaptive=adaptive,
    )
    args = build_dqn_args(
        results_dir=results_dir,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes_during_training,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=default_device,
        hyperparameters=hyperparameters,
        extra=attention_config if attention else None,
    )

    display(pd.DataFrame({"training_and_eval_env": pd.Series(env_config)}))
    summary = train_and_display(
        trainer,
        args,
        env_config,
        label=label,
    )
    saved_eval_summary = evaluate_saved_model(
        trainer,
        summary_path=results_dir / run_name / "summary.json",
        env_config=env_config,
        episodes=saved_model_eval_episodes,
        seed=saved_model_eval_seed + seed_offset,
        name=saved_model_eval_name,
        label=label,
    )
    output = {
        "run_name": run_name,
        "label": label,
        "backend_module": backend_module,
        "safety_reward": bool(safety_reward),
        "adaptive": bool(adaptive),
        "attention": bool(attention),
        "summary": summary,
        "saved_eval_summary": saved_eval_summary,
    }
    completed_experiments[run_name] = output
    return output

## Experiment 1: Baseline DQN

In [ ]:
experiment_1_baseline_dqn = run_congested_experiment(
    run_name="congested_baseline_dqn_20k",
    label="Congested Traffic - Baseline DQN",
    backend_module="elurant_dqn",
    seed_offset=0,
)

## Experiment 2: Baseline DQN + Safety Reward

In [ ]:
experiment_2_baseline_safety = run_congested_experiment(
    run_name="congested_baseline_dqn_safety_reward_20k",
    label="Congested Traffic - Baseline DQN + Safety Reward",
    backend_module="elurant_dqn",
    safety_reward=True,
    seed_offset=100,
)

## Experiment 3: Attention DQN

In [ ]:
experiment_3_attention_dqn = run_congested_experiment(
    run_name="congested_attention_dqn_20k",
    label="Congested Traffic - Attention DQN",
    backend_module="attention_dqn",
    attention=True,
    seed_offset=200,
)

## Experiment 4: Attention DQN + Safety Reward

In [ ]:
experiment_4_attention_safety = run_congested_experiment(
    run_name="congested_attention_dqn_safety_reward_20k",
    label="Congested Traffic - Attention DQN + Safety Reward",
    backend_module="attention_dqn",
    attention=True,
    safety_reward=True,
    seed_offset=300,
)

## Experiment 5: Adaptive TTC Wide-Band DQN

In [ ]:
experiment_5_adaptive_wide_band = run_congested_experiment(
    run_name="congested_adaptive_wide_band_20k",
    label="Congested Traffic - Adaptive TTC Wide-Band DQN",
    backend_module="elurant_dqn",
    adaptive=True,
    seed_offset=400,
)

## Experiment 6: Adaptive TTC Wide-Band DQN + Safety Reward

In [ ]:
experiment_6_adaptive_wide_band_safety = run_congested_experiment(
    run_name="congested_adaptive_wide_band_safety_reward_20k",
    label="Congested Traffic - Adaptive TTC Wide-Band DQN + Safety Reward",
    backend_module="elurant_dqn",
    adaptive=True,
    safety_reward=True,
    seed_offset=500,
)

## Experiment 7: Adaptive TTC Wide-Band Attention DQN

In [ ]:
experiment_7_adaptive_wide_band_attention = run_congested_experiment(
    run_name="congested_adaptive_wide_band_attention_20k",
    label="Congested Traffic - Adaptive TTC Wide-Band Attention DQN",
    backend_module="attention_dqn",
    attention=True,
    adaptive=True,
    seed_offset=600,
)

## Collision Diagnostics

Run saved congested-traffic models through action- and lane-level diagnostics. This does not retrain models; it loads existing `summary.json` / `model_path` artifacts and labels each episode by likely failure mode.

In [ ]:
from stable_baselines3 import DQN

DQN_SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "DQN"
dqn_script_dir_str = str(DQN_SCRIPT_DIR)
if dqn_script_dir_str not in sys.path:
    sys.path.insert(0, dqn_script_dir_str)

from congestion_diagnostics import (
    evaluate_congestion_diagnostics,
    save_congestion_diagnostics,
)

CONGESTED_RUN_SPECS = [
    {"run_name": "congested_baseline_dqn_20k", "label": "Baseline DQN", "backend_module": "elurant_dqn", "seed_offset": 0},
    {"run_name": "congested_baseline_dqn_safety_reward_20k", "label": "Baseline DQN + Safety Reward", "backend_module": "elurant_dqn", "safety_reward": True, "seed_offset": 100},
    {"run_name": "congested_attention_dqn_20k", "label": "Attention DQN", "backend_module": "attention_dqn", "attention": True, "seed_offset": 200},
    {"run_name": "congested_attention_dqn_safety_reward_20k", "label": "Attention DQN + Safety Reward", "backend_module": "attention_dqn", "attention": True, "safety_reward": True, "seed_offset": 300},
    {"run_name": "congested_adaptive_wide_band_20k", "label": "Adaptive TTC Wide-Band DQN", "backend_module": "elurant_dqn", "adaptive": True, "seed_offset": 400},
    {"run_name": "congested_adaptive_wide_band_safety_reward_20k", "label": "Adaptive TTC Wide-Band DQN + Safety Reward", "backend_module": "elurant_dqn", "adaptive": True, "safety_reward": True, "seed_offset": 500},
    {"run_name": "congested_adaptive_wide_band_attention_20k", "label": "Adaptive TTC Wide-Band Attention DQN", "backend_module": "attention_dqn", "attention": True, "adaptive": True, "seed_offset": 600},
]

DIAGNOSTIC_EPISODES = 100
DIAGNOSTIC_SEED = seed + 50_000
DIAGNOSTIC_CONFIG = {
    "front_ttc_safe": 4.0,
    "front_ttc_critical": 1.5,
    "rear_ttc_safe": 4.0,
    "rear_ttc_critical": 1.5,
    "lane_gap_safe": 12.0,
    "bad_action_margin": 0.35,
    "no_good_action_risk": 0.85,
}


def diagnostic_failure_reason(row: pd.Series) -> str:
    if not bool(row.get("collision", False)):
        return "no_collision"
    if bool(row.get("agent_chose_badly", False)):
        return "bad_final_action"
    if bool(row.get("wrong_lane_earlier", False)):
        return "missed_better_lane"
    if bool(row.get("unavoidable_rear_pressure", False)):
        return "rear_pressure_no_good_action"
    if bool(row.get("no_good_discrete_action", False)):
        return "no_good_discrete_action"
    return f"collision_{row.get('collision_type', 'unknown')}"


def run_collision_diagnostics_for_spec(spec: dict, episodes: int = DIAGNOSTIC_EPISODES) -> dict | None:
    run_name = spec["run_name"]
    run_dir = RESULTS_DIR / run_name
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        print(f"Skipping {run_name}: missing {summary_path}")
        return None

    trainer, _, _, _, default_device = load_dqn_backend(
        backend_module=spec["backend_module"],
        notebook_subdir="congested_traffic_policy",
        results_subdir=RESULTS_SUBDIR,
    )
    run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    model_path = Path(run_summary["model_path"])
    if not model_path.exists():
        print(f"Skipping {run_name}: missing model {model_path}")
        return None

    env_config = make_congested_env_config(
        safety_reward=bool(spec.get("safety_reward", False)),
        adaptive=bool(spec.get("adaptive", False)),
    )
    print(f"Running collision diagnostics for {run_name} ({episodes} episodes)")
    model = DQN.load(str(model_path), device=default_device)
    diagnostic_df, diagnostic_traces = evaluate_congestion_diagnostics(
        model,
        trainer.make_env,
        env_config=env_config,
        episodes=episodes,
        seed=DIAGNOSTIC_SEED + int(spec.get("seed_offset", 0)),
        diagnostic_config=DIAGNOSTIC_CONFIG,
    )
    diagnostic_df["run_name"] = run_name
    diagnostic_df["label"] = spec["label"]
    diagnostic_df["failure_reason"] = diagnostic_df.apply(diagnostic_failure_reason, axis=1)

    output_dir = run_dir / f"collision_diagnostics_{episodes}_episodes"
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = save_congestion_diagnostics(diagnostic_df, diagnostic_traces, output_dir)
    enriched_path = output_dir / "collision_diagnostic_summary_enriched.csv"
    diagnostic_df.to_csv(enriched_path, index=False)

    failure_counts = (
        diagnostic_df["failure_reason"]
        .value_counts(dropna=False)
        .rename_axis("failure_reason")
        .reset_index(name="episodes")
    )
    failure_counts["rate"] = failure_counts["episodes"] / max(len(diagnostic_df), 1)
    failure_counts.to_csv(output_dir / "failure_reason_counts.csv", index=False)

    return {
        "run_name": run_name,
        "label": spec["label"],
        "episodes": int(episodes),
        "collision_rate": float(diagnostic_df["collision"].mean()) if len(diagnostic_df) else 0.0,
        "bad_final_action_rate": float(diagnostic_df["agent_chose_badly"].mean()) if len(diagnostic_df) else 0.0,
        "no_good_action_rate": float(diagnostic_df["no_good_discrete_action"].mean()) if len(diagnostic_df) else 0.0,
        "missed_better_lane_rate": float(diagnostic_df["wrong_lane_earlier"].mean()) if len(diagnostic_df) else 0.0,
        "unavoidable_rear_pressure_rate": float(diagnostic_df["unavoidable_rear_pressure"].mean()) if len(diagnostic_df) else 0.0,
        "mean_bad_action_count": float(diagnostic_df["bad_action_count"].mean()) if len(diagnostic_df) else 0.0,
        "mean_missed_better_lane_count": float(diagnostic_df["missed_better_lane_count"].mean()) if len(diagnostic_df) else 0.0,
        "summary_path": paths["summary_path"],
        "traces_path": paths["traces_path"],
        "enriched_csv_path": str(enriched_path),
        "failure_counts_path": str(output_dir / "failure_reason_counts.csv"),
    }


In [ ]:
diagnostic_outputs = []
for spec in CONGESTED_RUN_SPECS:
    output = run_collision_diagnostics_for_spec(spec, episodes=DIAGNOSTIC_EPISODES)
    if output is not None:
        diagnostic_outputs.append(output)

collision_diagnostic_comparison_df = pd.DataFrame(diagnostic_outputs)
comparison_diagnostic_path = RESULTS_DIR / f"collision_diagnostic_comparison_{DIAGNOSTIC_EPISODES}_episodes.csv"
collision_diagnostic_comparison_df.to_csv(comparison_diagnostic_path, index=False)
print("Collision diagnostic comparison saved to:", comparison_diagnostic_path)
display(collision_diagnostic_comparison_df.round(3))


In [ ]:
# Failure-mode matrix across all diagnostic runs.
failure_frames = []
for output in diagnostic_outputs:
    counts_path = Path(output["failure_counts_path"])
    if counts_path.exists():
        counts = pd.read_csv(counts_path)
        counts["run_name"] = output["run_name"]
        counts["label"] = output["label"]
        failure_frames.append(counts)

if failure_frames:
    failure_reason_df = pd.concat(failure_frames, ignore_index=True)
    failure_reason_matrix = (
        failure_reason_df.pivot_table(
            index=["run_name", "label"],
            columns="failure_reason",
            values="rate",
            fill_value=0.0,
        )
        .reset_index()
    )
    matrix_path = RESULTS_DIR / f"collision_failure_reason_matrix_{DIAGNOSTIC_EPISODES}_episodes.csv"
    failure_reason_matrix.to_csv(matrix_path, index=False)
    print("Failure reason matrix saved to:", matrix_path)
    display(failure_reason_matrix.round(3))
else:
    print("No diagnostic failure-count files found. Run the diagnostics cell first.")


## Comparison Table

In [ ]:
# Aggregate the saved 1k evaluation summaries for all runs that have completed.
rows = []
for run_name, output in completed_experiments.items():
    summary_records = output["saved_eval_summary"].get("metric_summary", [])
    row = {
        "run_name": run_name,
        "label": output["label"],
        "attention": output["attention"],
        "adaptive": output["adaptive"],
        "safety_reward": output["safety_reward"],
    }
    for record in summary_records:
        metric = record.get("metric", "").replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_per_").lower()
        row[f"{metric}_mean"] = record.get("mean")
        row[f"{metric}_std"] = record.get("std")
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_path = RESULTS_DIR / "congested_policy_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Comparison saved to:", comparison_path)
display(comparison_df.round(3))